# Dollar Volume Edge — Backtest Riguroso sobre Small Caps

**Hipótesis:** El ratio de Dollar Volume acumulado intradiario respecto al Day T genera edge en continuación, agotamiento y rebote.

**Universo:** Tickers que aparecen persistentemente (>= 3 snapshots) en la categoría 'Top Gainers' de finviz-dashboard con >=15% de cambio en el día.

**Datos:** `finviz_snapshots.db` — tabla `snapshots` (universo Day T) + tabla `market_bars` (1-min OHLCV de IBKR).

**Metodología:** Walk-forward sin lookahead. Slippage + comisiones realistas. Métricas profesionales por bucket de ratio.

---
**Diseño de las 3 estrategias:**

| Estrategia | Condición de entrada | Dirección | Hipótesis |
|---|---|---|---|
| **Continuation Long** | DV_ratio T+1 < 0.5, precio > VWAP, primera hora | Long | El momentum no se ha agotado |
| **Exhaustion Short** | DV_ratio T+1 > 1.0, precio < VWAP, lower high | Short | El volumen se agotó, reversión |
| **Bounce Long** | DV_ratio > 1.5 en T, T+1 open < close_T * 0.97, precio toca VWAP | Long | Mean reversion post-agotamiento |

**Stop:** ATR(14) x 1.5 o nivel técnico (low de la barra de entrada)
**Target:** 2R fijo
**Slippage:** 0.05% por lado
**Comisión:** $0.005 por acción

In [ ]:
import sqlite3
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from datetime import datetime, timedelta
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 11

# ─────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────
FINVIZ_DB = os.path.expanduser(
    '~/Library/Application Support/finviz-dashboard/finviz_snapshots.db'
)

# Filtros para Day T candidates (igual que finviz_to_proactive.py)
MIN_APPEARANCES  = 3      # apariciones mínimas en snapshots del día
MIN_CHANGE_PCT   = 15.0   # % mínimo de subida en Day T
MIN_PRICE        = 0.80   # precio mínimo
MAX_PRICE        = 50.0   # precio máximo

# Parámetros de backtest
SLIPPAGE_PCT        = 0.0005   # 0.05% por lado
COMMISSION_PER_SHARE = 0.005   # IBKR tiered
SHARES_PER_TRADE    = 100      # tamaño fijo para P&L en $
STOP_ATR_MULT       = 1.5      # stop = ATR(14) * mult
TARGET_R            = 2.0      # target = 2R
MAX_HOLD_BARS       = 60       # máximo 60 barras de 1-min (~1h)

# Buckets de ratio DV
RATIO_BINS   = [0, 0.3, 0.6, 1.0, 1.5, np.inf]
RATIO_LABELS = ['0-0.3x', '0.3-0.6x', '0.6-1.0x', '1.0-1.5x', '>1.5x']

# Horario de mercado (barras válidas)
MARKET_OPEN_HOUR  = 9
MARKET_OPEN_MIN   = 30
MARKET_CLOSE_HOUR = 16
MARKET_CLOSE_MIN  = 0

# TWS (opcional, para descargar barras faltantes)
TWS_HOST     = '127.0.0.1'
TWS_PORT     = 7497
TWS_CLIENT_ID = 5055
USE_TWS      = False   # cambiar a True si TWS está corriendo

print(f'DB path: {FINVIZ_DB}')
print(f'DB exists: {os.path.exists(FINVIZ_DB)}')

## 1. Carga del universo Day T desde finviz_snapshots.db

In [ ]:
def _parse_change(v) -> float:
    try:
        return float(str(v).replace('%', '').replace('+', '').strip())
    except Exception:
        return 0.0

def _parse_price(v) -> float:
    try:
        return float(str(v).replace('$', '').replace(',', '').strip())
    except Exception:
        return 0.0

def load_day_t_candidates(db_path: str) -> pd.DataFrame:
    """
    Carga los Day T candidates desde la tabla snapshots.
    Replica la lógica de get_persistent_gainers() de finviz_to_proactive.py.
    Retorna un DataFrame con: ticker, day_t_date, appearances, max_change_pct,
                               last_price, first_seen, last_seen
    """
    if not os.path.exists(db_path):
        raise FileNotFoundError(f'DB no encontrada: {db_path}')

    with sqlite3.connect(db_path) as conn:
        rows = conn.execute("""
            SELECT ticker, price, change_pct, volume, timestamp
            FROM snapshots
            WHERE category = 'Top Gainers'
            ORDER BY ticker, timestamp
        """).fetchall()

    # Agrupar por (ticker, day)
    from collections import defaultdict
    by_ticker_day = defaultdict(list)
    for ticker, price, change_pct, volume, ts in rows:
        day = ts[:10]
        by_ticker_day[(ticker, day)].append({
            'price': price, 'change_pct': change_pct,
            'volume': volume, 'timestamp': ts
        })

    result = []
    for (ticker, day), snapshots in by_ticker_day.items():
        appearances = len(set(s['timestamp'] for s in snapshots))
        if appearances < MIN_APPEARANCES:
            continue

        changes = [_parse_change(s['change_pct']) for s in snapshots]
        prices  = [_parse_price(s['price']) for s in snapshots]

        max_change = max(changes)
        if max_change < MIN_CHANGE_PCT:
            continue

        last_price = prices[-1]
        if last_price < MIN_PRICE or last_price > MAX_PRICE:
            continue

        result.append({
            'ticker':       ticker,
            'day_t_date':   day,
            'appearances':  appearances,
            'max_change_pct': max_change,
            'last_price':   last_price,
            'first_seen':   snapshots[0]['timestamp'],
            'last_seen':    snapshots[-1]['timestamp'],
        })

    df = pd.DataFrame(result)
    if df.empty:
        print('WARN: No Day T candidates found. Revisa los filtros o la DB.')
        return df

    df['day_t_date'] = pd.to_datetime(df['day_t_date'])
    df = df.sort_values(['day_t_date', 'max_change_pct'], ascending=[False, False])
    df = df.reset_index(drop=True)
    return df


candidates = load_day_t_candidates(FINVIZ_DB)
print(f'Day T candidates: {len(candidates)} entradas, {candidates["ticker"].nunique()} tickers únicos')
print(f'Rango de fechas: {candidates["day_t_date"].min().date()} → {candidates["day_t_date"].max().date()}')
print()
candidates.head(15)

## 2. Carga de barras 1-min desde market_bars

In [ ]:
def load_market_bars(db_path: str, tickers: list, dates: list) -> pd.DataFrame:
    """
    Carga barras 1-min de la tabla market_bars.
    tickers: lista de símbolos
    dates:   lista de fechas como strings 'YYYY-MM-DD'
    Retorna DataFrame con: ticker, timestamp (naive ET), open, high, low, close, volume
    """
    if not os.path.exists(db_path):
        return pd.DataFrame()

    with sqlite3.connect(db_path) as conn:
        # Verificar si existe la tabla
        tables = [r[0] for r in conn.execute(
            "SELECT name FROM sqlite_master WHERE type='table'"
        ).fetchall()]
        if 'market_bars' not in tables:
            print('WARN: tabla market_bars no existe en la DB.')
            return pd.DataFrame()

        # Construir placeholders
        tick_ph  = ','.join('?' * len(tickers))
        date_filters = []
        params = list(tickers)
        for d in dates:
            date_filters.append("bar_time LIKE ?")
            params.append(f"{d}%")
        date_clause = ' OR '.join(date_filters)

        query = f"""
            SELECT ticker, bar_time, open, high, low, close, volume
            FROM market_bars
            WHERE ticker IN ({tick_ph})
              AND ({date_clause})
            ORDER BY ticker, bar_time
        """
        rows = conn.execute(query, params).fetchall()

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows, columns=['ticker', 'timestamp', 'open', 'high', 'low', 'close', 'volume'])
    # bar_time es string naive ET — parsear directamente
    df['timestamp'] = pd.to_datetime(df['timestamp'].str[:19])
    df['date'] = df['timestamp'].dt.date

    # Filtrar solo horario de mercado (9:30 - 16:00)
    t = df['timestamp']
    market_open  = (t.dt.hour > MARKET_OPEN_HOUR) | ((t.dt.hour == MARKET_OPEN_HOUR) & (t.dt.minute >= MARKET_OPEN_MIN))
    market_close = (t.dt.hour < MARKET_CLOSE_HOUR)
    df = df[market_open & market_close].reset_index(drop=True)

    for col in ['open', 'high', 'low', 'close', 'volume']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    return df.dropna(subset=['close', 'volume'])


def fetch_tws_bars(ticker: str, date_str: str) -> pd.DataFrame:
    """
    Descarga barras 1-min de TWS para un ticker/fecha y las guarda en market_bars.
    Requiere USE_TWS=True y TWS corriendo.
    """
    if not USE_TWS:
        return pd.DataFrame()
    try:
        from ib_insync import IB, Stock, util
        ib = IB()
        ib.connect(TWS_HOST, TWS_PORT, clientId=TWS_CLIENT_ID)
        contract = Stock(ticker, 'SMART', 'USD')
        ib.qualifyContracts(contract)
        end_dt = f"{date_str} 16:00:00 US/Eastern"
        bars = ib.reqHistoricalData(
            contract, endDateTime=end_dt,
            durationStr='1 D', barSizeSetting='1 min',
            whatToShow='TRADES', useRTH=True, formatDate=1
        )
        ib.disconnect()
        if not bars:
            return pd.DataFrame()
        df = util.df(bars)
        df = df.rename(columns={'date': 'timestamp'})
        df['ticker'] = ticker
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df['date'] = df['timestamp'].dt.date

        # Guardar en market_bars
        with sqlite3.connect(FINVIZ_DB) as conn:
            conn.execute("""
                CREATE TABLE IF NOT EXISTS market_bars (
                    ticker TEXT, bar_time TEXT, open REAL, high REAL,
                    low REAL, close REAL, volume INTEGER,
                    PRIMARY KEY (ticker, bar_time)
                )
            """)
            for _, row in df.iterrows():
                conn.execute("""
                    INSERT OR IGNORE INTO market_bars
                    (ticker, bar_time, open, high, low, close, volume)
                    VALUES (?, ?, ?, ?, ?, ?, ?)
                """, (ticker, str(row['timestamp'])[:19],
                      row['open'], row['high'], row['low'],
                      row['close'], int(row['volume'])))
            conn.commit()

        return df[['ticker', 'timestamp', 'date', 'open', 'high', 'low', 'close', 'volume']]

    except Exception as e:
        print(f'TWS error for {ticker} {date_str}: {e}')
        return pd.DataFrame()


# Determinar qué fechas necesitamos: Day T y T+1 y T+2
if candidates.empty:
    bars_df = pd.DataFrame()
    print('Sin candidates — sin barras.')
else:
    needed_tickers = candidates['ticker'].unique().tolist()

    # Generar fechas D, D+1, D+2 para cada candidate
    needed_dates = set()
    import pandas.tseries.offsets as offsets
    for _, row in candidates.iterrows():
        d = row['day_t_date']
        for i in range(3):
            nd = d + pd.offsets.BDay(i)
            needed_dates.add(nd.strftime('%Y-%m-%d'))
    needed_dates = sorted(needed_dates)

    bars_df = load_market_bars(FINVIZ_DB, needed_tickers, needed_dates)
    print(f'Barras cargadas desde market_bars: {len(bars_df)}')

    if USE_TWS and not bars_df.empty:
        # Detectar qué combinaciones (ticker, date) faltan
        existing = set(zip(bars_df['ticker'].astype(str), bars_df['date'].astype(str)))
        missing = []
        for t in needed_tickers:
            for d in needed_dates:
                if (t, d) not in existing:
                    missing.append((t, d))
        print(f'Combinaciones (ticker,date) faltantes: {len(missing)} — descargando de TWS...')
        extra = []
        for t, d in missing[:50]:  # límite de 50 para no saturar TWS
            df_extra = fetch_tws_bars(t, d)
            if not df_extra.empty:
                extra.append(df_extra)
        if extra:
            bars_df = pd.concat([bars_df] + extra, ignore_index=True)
            print(f'Barras totales tras TWS: {len(bars_df)}')

print(f'\nTickers con barras: {bars_df["ticker"].nunique() if not bars_df.empty else 0}')

## 3. Feature Engineering — VWAP, ATR, DV ratio

In [ ]:
def compute_features(bars: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula por barra:
    - dollar_volume: close * volume
    - cum_dv: DV acumulado intradiario
    - vwap: VWAP intradiario
    - atr14: ATR 14 periodos (sobre barras del mismo día)
    - bar_index: numero de barra del dia (0 = primera barra 9:30)
    """
    if bars.empty:
        return bars

    bars = bars.sort_values(['ticker', 'timestamp']).copy()

    # Dollar volume
    bars['dollar_volume'] = bars['close'] * bars['volume']

    # Cumulative DV y VWAP intradiarios
    grp = bars.groupby(['ticker', 'date'])
    bars['cum_dv']     = grp['dollar_volume'].cumsum()
    bars['cum_vol']    = grp['volume'].cumsum()
    bars['cum_tp_vol'] = grp.apply(
        lambda g: ((g['high'] + g['low'] + g['close']) / 3 * g['volume']).cumsum()
    ).reset_index(level=[0, 1], drop=True)
    bars['vwap'] = bars['cum_tp_vol'] / bars['cum_vol']

    # ATR 14 (True Range rolling dentro del día)
    bars['prev_close'] = bars.groupby(['ticker', 'date'])['close'].shift(1)
    bars['tr'] = bars.apply(
        lambda r: max(
            r['high'] - r['low'],
            abs(r['high'] - r['prev_close']) if not pd.isna(r['prev_close']) else 0,
            abs(r['low']  - r['prev_close']) if not pd.isna(r['prev_close']) else 0,
        ), axis=1
    )
    bars['atr14'] = grp['tr'].transform(lambda x: x.rolling(14, min_periods=1).mean())

    # Bar index dentro del día
    bars['bar_index'] = grp.cumcount()

    # Indicadores de vela
    bars['is_green'] = bars['close'] > bars['open']
    bars['is_red']   = bars['close'] < bars['open']

    return bars.drop(columns=['prev_close', 'tr', 'cum_tp_vol', 'cum_vol'], errors='ignore')


def attach_day_t_dv(bars: pd.DataFrame, candidates: pd.DataFrame) -> pd.DataFrame:
    """
    Para cada barra posterior a Day T, adjunta:
    - day_t_total_dv: DV total del Day T (suma de todas las barras del day T)
    - dv_ratio: cum_dv / day_t_total_dv
    - days_since_t: dias hábiles desde Day T (1 = T+1, 2 = T+2, ...)
    """
    if bars.empty or candidates.empty:
        return bars

    # Computar DV total por (ticker, date)
    daily_dv = bars.groupby(['ticker', 'date'])['dollar_volume'].sum().reset_index()
    daily_dv.columns = ['ticker', 'date', 'total_day_dv']
    daily_dv['date'] = pd.to_datetime(daily_dv['date'])

    # Mapeo: para cada candidate, adjuntar day_t_dv a filas T+1, T+2...
    bars['date_ts'] = pd.to_datetime(bars['date'])

    # Crear lookup {(ticker, day_t_date) -> total_dv}
    cand_lookup = {}
    for _, row in candidates.iterrows():
        t = row['ticker']
        d = row['day_t_date']
        # buscar en daily_dv
        match = daily_dv[(daily_dv['ticker'] == t) & (daily_dv['date'] == d)]
        if not match.empty:
            cand_lookup[(t, d)] = match.iloc[0]['total_day_dv']
        else:
            # Estimar desde last_price y last_volume del candidate
            cand_lookup[(t, d)] = row['last_price'] * 500_000  # fallback grosero

    # Para cada barra, encontrar el day_t más reciente para ese ticker
    candidate_dates = candidates.sort_values('day_t_date')

    day_t_dv_col    = []
    days_since_t_col = []

    import pandas.tseries.offsets as offsets
    bday = offsets.BDay()

    for _, bar_row in bars.iterrows():
        ticker = bar_row['ticker']
        bar_date = bar_row['date_ts']

        # Filtrar candidates del mismo ticker con day_t_date < bar_date
        tc = candidate_dates[(candidate_dates['ticker'] == ticker) &
                             (candidate_dates['day_t_date'] < bar_date)]

        if tc.empty:
            day_t_dv_col.append(np.nan)
            days_since_t_col.append(np.nan)
            continue

        # Tomar el Day T más reciente
        last_day_t = tc.iloc[-1]['day_t_date']
        dv_ref = cand_lookup.get((ticker, last_day_t), np.nan)

        # Días hábiles desde Day T
        try:
            bd_range = pd.bdate_range(last_day_t, bar_date)
            days_since = len(bd_range) - 1  # -1 porque incluye el propio last_day_t
        except Exception:
            days_since = np.nan

        day_t_dv_col.append(dv_ref)
        days_since_t_col.append(days_since)

    bars['day_t_total_dv']  = day_t_dv_col
    bars['days_since_t']    = days_since_t_col
    bars['dv_ratio']        = bars['cum_dv'] / bars['day_t_total_dv']

    # Bucket de ratio
    bars['ratio_bucket'] = pd.cut(
        bars['dv_ratio'], bins=RATIO_BINS, labels=RATIO_LABELS, right=False
    )

    bars = bars.drop(columns=['date_ts'], errors='ignore')
    return bars


if not bars_df.empty:
    print('Calculando features...')
    bars_feat = compute_features(bars_df)
    print('Adjuntando Day T DV reference...')
    bars_feat = attach_day_t_dv(bars_feat, candidates)

    # Solo filas con dv_ratio válido y days_since_t en [1, 3]
    tradeable = bars_feat[
        bars_feat['dv_ratio'].notna() &
        bars_feat['days_since_t'].between(1, 3)
    ].copy()
    print(f'Barras tradeables (T+1 a T+3 con dv_ratio): {len(tradeable)}')
    print(f'Tickers: {tradeable["ticker"].nunique()}')
else:
    bars_feat = pd.DataFrame()
    tradeable = pd.DataFrame()
    print('Sin barras — omitiendo feature engineering.')

## 4. Backtest Engine

Reglas estrictas:
- Entry: precio de la siguiente barra después de la señal (no lookahead)
- Stop: entry - ATR(14) * STOP_ATR_MULT (para longs), +ATR para shorts
- Target: entry + 2 * riesgo (2R)
- Exit al EOD si no se toca stop ni target
- Slippage aplicado al entry y al exit

In [ ]:
def simulate_trade(
    bars_day: pd.DataFrame,
    signal_bar_idx: int,      # índice de la barra donde se genera la señal
    direction: str,           # 'long' o 'short'
    atr_at_signal: float,
) -> dict | None:
    """
    Simula una operación a partir del bar siguiente a signal_bar_idx.
    Retorna dict con métricas o None si no se puede ejecutar.

    IMPORTANT: La señal se genera EN signal_bar_idx. El entry es la apertura
    de signal_bar_idx + 1. Esto elimina el lookahead de 1 barra.
    """
    # Siguiente barra
    entry_bar_pos = signal_bar_idx + 1
    if entry_bar_pos >= len(bars_day):
        return None  # señal en la última barra — no hay entrada posible

    entry_row = bars_day.iloc[entry_bar_pos]
    raw_entry = entry_row['open']  # entry a la apertura de la barra siguiente

    if direction == 'long':
        entry_price = raw_entry * (1 + SLIPPAGE_PCT)
        stop_price  = entry_price - (atr_at_signal * STOP_ATR_MULT)
        target_price = entry_price + TARGET_R * (entry_price - stop_price)
        if stop_price <= 0 or stop_price >= entry_price:
            return None
    else:  # short
        entry_price  = raw_entry * (1 - SLIPPAGE_PCT)
        stop_price   = entry_price + (atr_at_signal * STOP_ATR_MULT)
        target_price = entry_price - TARGET_R * (stop_price - entry_price)
        if target_price <= 0 or stop_price <= entry_price:
            return None

    risk_per_share = abs(entry_price - stop_price)
    if risk_per_share < 0.001:
        return None

    # Simular barra a barra
    exit_price  = None
    exit_reason = 'EOD'
    exit_bar    = len(bars_day) - 1
    max_hold    = min(entry_bar_pos + MAX_HOLD_BARS, len(bars_day))

    for i in range(entry_bar_pos, max_hold):
        row = bars_day.iloc[i]
        if direction == 'long':
            if row['low'] <= stop_price:
                exit_price  = stop_price * (1 - SLIPPAGE_PCT)
                exit_reason = 'STOP'
                exit_bar    = i
                break
            if row['high'] >= target_price:
                exit_price  = target_price * (1 - SLIPPAGE_PCT)
                exit_reason = 'TARGET'
                exit_bar    = i
                break
        else:  # short
            if row['high'] >= stop_price:
                exit_price  = stop_price * (1 + SLIPPAGE_PCT)
                exit_reason = 'STOP'
                exit_bar    = i
                break
            if row['low'] <= target_price:
                exit_price  = target_price * (1 + SLIPPAGE_PCT)
                exit_reason = 'TARGET'
                exit_bar    = i
                break

    if exit_price is None:
        # Salida a la última barra válida (MAX_HOLD o EOD)
        last_bar = bars_day.iloc[min(max_hold - 1, len(bars_day) - 1)]
        if direction == 'long':
            exit_price = last_bar['close'] * (1 - SLIPPAGE_PCT)
        else:
            exit_price = last_bar['close'] * (1 + SLIPPAGE_PCT)
        exit_bar    = max_hold - 1
        exit_reason = 'TIME' if max_hold < len(bars_day) else 'EOD'

    commission = COMMISSION_PER_SHARE * 2  # entry + exit por acción

    if direction == 'long':
        gross_pnl_per_share = exit_price - entry_price
    else:
        gross_pnl_per_share = entry_price - exit_price

    net_pnl_per_share = gross_pnl_per_share - commission
    r_multiple = net_pnl_per_share / risk_per_share
    net_pnl    = net_pnl_per_share * SHARES_PER_TRADE

    return {
        'entry_price':  entry_price,
        'exit_price':   exit_price,
        'stop_price':   stop_price,
        'target_price': target_price,
        'exit_reason':  exit_reason,
        'hold_bars':    exit_bar - entry_bar_pos,
        'r_multiple':   r_multiple,
        'net_pnl':      net_pnl,
        'risk_per_share': risk_per_share,
    }


print('simulate_trade() definida OK')

In [ ]:
def run_strategy_continuation_long(tradeable: pd.DataFrame) -> pd.DataFrame:
    """
    Continuation Long:
    - Operamos T+1 únicamente (days_since_t == 1)
    - Señal: primera barra donde DV_ratio < 0.5 Y close > VWAP Y bar_index entre 3 y 60
    - 1 trade por ticker/día (primera señal)
    """
    trades = []
    subset = tradeable[tradeable['days_since_t'] == 1].copy()

    for (ticker, date), day_data in subset.groupby(['ticker', 'date']):
        day_data = day_data.reset_index(drop=True)

        # Señal: primera barra que cumple condiciones (sin lookahead — la señal se ejecuta en la siguiente)
        signal_rows = day_data[
            (day_data['dv_ratio'] < 0.5) &
            (day_data['close'] > day_data['vwap']) &
            (day_data['bar_index'] >= 3) &
            (day_data['bar_index'] <= 60) &
            (day_data['is_green'])  # vela verde para momentum
        ]
        if signal_rows.empty:
            continue

        sig_idx = signal_rows.index[0]
        sig_row = day_data.iloc[sig_idx]
        atr     = sig_row['atr14']
        if atr <= 0 or pd.isna(atr):
            continue

        result = simulate_trade(day_data, sig_idx, 'long', atr)
        if result is None:
            continue

        trades.append({
            'ticker':      ticker,
            'date':        date,
            'strategy':    'Continuation_Long',
            'signal_bar':  sig_idx,
            'dv_ratio_at_signal': sig_row['dv_ratio'],
            'ratio_bucket': sig_row['ratio_bucket'],
            'days_since_t': 1,
            **result,
        })

    return pd.DataFrame(trades)


def run_strategy_exhaustion_short(tradeable: pd.DataFrame) -> pd.DataFrame:
    """
    Exhaustion Short:
    - Operamos T+1 (days_since_t == 1)
    - Señal: primera barra donde DV_ratio > 1.0 Y close < VWAP Y lower high vs barra anterior
    - Indica que el volumen acumulado ya supera al día de explosión → reversión
    """
    trades = []
    subset = tradeable[tradeable['days_since_t'] == 1].copy()

    for (ticker, date), day_data in subset.groupby(['ticker', 'date']):
        day_data = day_data.reset_index(drop=True)
        prev_high = day_data['high'].shift(1)

        signal_rows = day_data[
            (day_data['dv_ratio'] > 1.0) &
            (day_data['close'] < day_data['vwap']) &
            (day_data['high'] < prev_high) &         # lower high
            (day_data['bar_index'] >= 5) &
            (day_data['bar_index'] <= 90) &
            (day_data['is_red'])                     # vela roja confirmando bajada
        ]
        if signal_rows.empty:
            continue

        sig_idx = signal_rows.index[0]
        sig_row = day_data.iloc[sig_idx]
        atr     = sig_row['atr14']
        if atr <= 0 or pd.isna(atr):
            continue

        result = simulate_trade(day_data, sig_idx, 'short', atr)
        if result is None:
            continue

        trades.append({
            'ticker':      ticker,
            'date':        date,
            'strategy':    'Exhaustion_Short',
            'signal_bar':  sig_idx,
            'dv_ratio_at_signal': sig_row['dv_ratio'],
            'ratio_bucket': sig_row['ratio_bucket'],
            'days_since_t': 1,
            **result,
        })

    return pd.DataFrame(trades)


def run_strategy_bounce_long(tradeable: pd.DataFrame, candidates: pd.DataFrame) -> pd.DataFrame:
    """
    Bounce / Mean Reversion Long:
    - Operamos T+1 o T+2 (days_since_t in [1, 2])
    - Condición de selección: Day T terminó con DV_ratio > 1.5 (dia de agotamiento)
    - Señal intradiaria: precio toca VWAP por primera vez después de haber estado por debajo
      Y el bar_index está entre 10 y 120 (evitar apertura caótica)
    """
    trades = []

    # Tickers donde Day T fue de alto agotamiento
    # Necesitamos el DV_ratio al final del Day T
    # Proxy: usamos day_t_total_dv / day_t_total_dv = 1.0 siempre al EOD
    # Mejor proxy: si cum_dv al cierre del Day T > 1.5 * day_t_total_dv, es raro
    # En realidad day_t_total_dv ES la suma del Day T, así que al final siempre ratio=1.0
    # El DV_ratio en T+1 compara el DV acumulado de T+1 contra el total del Day T
    # Si en T+1 el DV ya supera 1.5 = el día de T+1 está siendo más activo que el Day T completo
    # Para Bounce queremos: Day T tuvo gap_down en T+1

    # Tickers con gap-down en T+1 (open T+1 < close T)
    subset = tradeable[tradeable['days_since_t'].isin([1, 2])].copy()

    for (ticker, date), day_data in subset.groupby(['ticker', 'date']):
        day_data = day_data.reset_index(drop=True)
        if len(day_data) < 15:
            continue

        first_bar = day_data.iloc[0]
        open_price = first_bar['open']

        # Encontrar el Day T para este ticker
        cand_match = candidates[
            (candidates['ticker'] == ticker) &
            (candidates['day_t_date'] < pd.Timestamp(date))
        ]
        if cand_match.empty:
            continue
        last_cand = cand_match.iloc[-1]
        day_t_close = last_cand['last_price']

        # Gap-down: open T+1 < day_t_close * 0.97
        if open_price >= day_t_close * 0.97:
            continue  # no hay gap-down suficiente

        # Señal: precio cruza VWAP desde abajo (bounce)
        # Buscamos barra donde close > vwap Y barra anterior close < vwap
        prev_close = day_data['close'].shift(1)
        prev_vwap  = day_data['vwap'].shift(1)

        signal_rows = day_data[
            (day_data['close'] > day_data['vwap']) &
            (prev_close < prev_vwap) &              # cruce desde abajo
            (day_data['bar_index'] >= 10) &
            (day_data['bar_index'] <= 120)
        ]
        if signal_rows.empty:
            continue

        sig_idx = signal_rows.index[0]
        sig_row = day_data.iloc[sig_idx]
        atr     = sig_row['atr14']
        if atr <= 0 or pd.isna(atr):
            continue

        result = simulate_trade(day_data, sig_idx, 'long', atr)
        if result is None:
            continue

        trades.append({
            'ticker':      ticker,
            'date':        date,
            'strategy':    'Bounce_Long',
            'signal_bar':  sig_idx,
            'dv_ratio_at_signal': sig_row['dv_ratio'],
            'ratio_bucket': sig_row['ratio_bucket'],
            'days_since_t': day_data.iloc[0]['days_since_t'],
            **result,
        })

    return pd.DataFrame(trades)


# Ejecutar estrategias
if not tradeable.empty:
    print('Ejecutando estrategias...')
    df_cont  = run_strategy_continuation_long(tradeable)
    df_exh   = run_strategy_exhaustion_short(tradeable)
    df_bounce = run_strategy_bounce_long(tradeable, candidates)

    all_trades = pd.concat([df_cont, df_exh, df_bounce], ignore_index=True)
    print(f'  Continuation Long: {len(df_cont)} trades')
    print(f'  Exhaustion Short:  {len(df_exh)} trades')
    print(f'  Bounce Long:       {len(df_bounce)} trades')
    print(f'  TOTAL:             {len(all_trades)} trades')
else:
    all_trades = pd.DataFrame()
    print('Sin datos para backtest.')

## 5. Métricas por Estrategia

In [ ]:
def compute_metrics(trades: pd.DataFrame, label: str = '') -> pd.DataFrame:
    """
    Calcula métricas completas por estrategia.
    """
    if trades.empty:
        return pd.DataFrame()

    rows = []
    for strat, grp in trades.groupby('strategy'):
        grp = grp.copy()
        n      = len(grp)
        wins   = grp[grp['r_multiple'] > 0]
        losses = grp[grp['r_multiple'] <= 0]

        win_rate   = len(wins) / n
        avg_r      = grp['r_multiple'].mean()
        median_r   = grp['r_multiple'].median()
        std_r      = grp['r_multiple'].std()

        avg_win  = wins['r_multiple'].mean()  if len(wins)   > 0 else 0
        avg_loss = losses['r_multiple'].mean() if len(losses) > 0 else 0

        expectancy = avg_r  # E[R] = win_rate * avg_win + loss_rate * avg_loss

        gross_wins   = wins['net_pnl'].sum()
        gross_losses = losses['net_pnl'].sum()
        profit_factor = abs(gross_wins / gross_losses) if gross_losses != 0 else np.inf

        cum_pnl = grp.sort_values('date')['net_pnl'].cumsum()
        running_max = cum_pnl.cummax()
        drawdown    = cum_pnl - running_max
        max_dd      = drawdown.min()

        sharpe = avg_r / std_r * np.sqrt(252) if std_r > 0 else 0

        # t-test: H0 = expectancy == 0
        from scipy import stats
        t_stat, p_value = stats.ttest_1samp(grp['r_multiple'], 0)

        rows.append({
            'Strategy':       strat,
            'N Trades':       n,
            'Win Rate':       f'{win_rate:.1%}',
            'Avg R':          f'{avg_r:+.3f}',
            'Median R':       f'{median_r:+.3f}',
            'Std R':          f'{std_r:.3f}',
            'Avg Win':        f'{avg_win:+.3f}',
            'Avg Loss':       f'{avg_loss:+.3f}',
            'Expectancy':     f'{expectancy:+.3f}R',
            'Profit Factor':  f'{profit_factor:.2f}',
            'Total P&L $':    f'${cum_pnl.iloc[-1]:+,.0f}',
            'Max DD $':       f'${max_dd:,.0f}',
            'Sharpe (ann.)':  f'{sharpe:.2f}',
            'p-value':        f'{p_value:.3f}',
            'Significant?':   'YES' if p_value < 0.05 else 'no',
        })

    return pd.DataFrame(rows)


if not all_trades.empty:
    metrics_df = compute_metrics(all_trades)
    print('=== MÉTRICAS POR ESTRATEGIA ===')
    display(metrics_df.set_index('Strategy').T)
else:
    print('Sin trades — no hay métricas que mostrar.')
    print()
    print('Posibles causas:')
    print('  1. No hay datos en market_bars para los tickers del universo')
    print('  2. Los filtros son demasiado estrictos')
    print('  3. USE_TWS=False y los datos no están cacheados')
    print()
    print('Tickers en candidates:', candidates['ticker'].unique().tolist() if not candidates.empty else '[]')

## 6. Análisis por Bucket de DV Ratio

In [ ]:
if not all_trades.empty:
    bucket_stats = all_trades.groupby(['strategy', 'ratio_bucket']).agg(
        n_trades=('r_multiple', 'count'),
        win_rate=('r_multiple', lambda x: (x > 0).mean()),
        avg_r=('r_multiple', 'mean'),
        median_r=('r_multiple', 'median'),
        std_r=('r_multiple', 'std'),
        total_pnl=('net_pnl', 'sum'),
    ).reset_index()

    print('=== RENDIMIENTO POR RATIO BUCKET ===')
    display(bucket_stats.round(3))

    # Gráfico
    strategies = all_trades['strategy'].unique()
    fig, axes = plt.subplots(1, len(strategies), figsize=(6 * len(strategies), 5), sharey=False)
    if len(strategies) == 1:
        axes = [axes]

    for ax, strat in zip(axes, strategies):
        sub = bucket_stats[bucket_stats['strategy'] == strat]
        colors = ['green' if v > 0 else 'red' for v in sub['avg_r']]
        bars = ax.bar(sub['ratio_bucket'].astype(str), sub['avg_r'], color=colors, alpha=0.7)
        ax.axhline(0, color='black', linewidth=0.8)
        ax.set_title(f'{strat}\nAvg R por DV Ratio Bucket')
        ax.set_xlabel('DV Ratio Bucket')
        ax.set_ylabel('Avg R')
        for bar, n in zip(bars, sub['n_trades']):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'n={n}', ha='center', va='bottom', fontsize=9)
        ax.tick_params(axis='x', rotation=30)

    plt.tight_layout()
    plt.show()
else:
    print('Sin trades — omitiendo análisis por bucket.')

## 7. Curvas de Capital y Distribución de R

In [ ]:
if not all_trades.empty:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    # 1. Equity curves por estrategia
    ax = axes[0, 0]
    for strat in all_trades['strategy'].unique():
        sub = all_trades[all_trades['strategy'] == strat].sort_values('date')
        ax.plot(range(len(sub)), sub['net_pnl'].cumsum(), label=strat, linewidth=2)
    ax.axhline(0, color='gray', linestyle='--', alpha=0.7)
    ax.set_title('Equity Curve ($) — por Estrategia')
    ax.set_xlabel('Trade #')
    ax.set_ylabel('P&L acumulado ($)')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # 2. Distribución de R-múltiplos (histograma)
    ax = axes[0, 1]
    for strat in all_trades['strategy'].unique():
        sub = all_trades[all_trades['strategy'] == strat]
        sub['r_multiple'].hist(ax=ax, bins=30, alpha=0.6, label=strat)
    ax.axvline(0, color='red', linestyle='--', linewidth=1.5)
    ax.set_title('Distribución de R-múltiplos')
    ax.set_xlabel('R-múltiplo')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # 3. Win Rate por estrategia
    ax = axes[1, 0]
    strat_wr = all_trades.groupby('strategy').apply(lambda x: (x['r_multiple'] > 0).mean()).reset_index()
    strat_wr.columns = ['strategy', 'win_rate']
    colors = ['green' if v >= 0.5 else 'orange' if v >= 0.4 else 'red' for v in strat_wr['win_rate']]
    bars = ax.bar(strat_wr['strategy'], strat_wr['win_rate'], color=colors, alpha=0.8)
    ax.axhline(0.5, color='gray', linestyle='--', alpha=0.7, label='50% ref')
    ax.set_title('Win Rate por Estrategia')
    ax.set_ylim(0, 1)
    ax.set_ylabel('Win Rate')
    for bar, v in zip(bars, strat_wr['win_rate']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{v:.1%}', ha='center', va='bottom', fontweight='bold')
    ax.legend()

    # 4. Exit reason breakdown
    ax = axes[1, 1]
    exit_counts = all_trades.groupby(['strategy', 'exit_reason']).size().unstack(fill_value=0)
    exit_counts.plot(kind='bar', ax=ax, alpha=0.8)
    ax.set_title('Motivo de Salida por Estrategia')
    ax.set_xlabel('Estrategia')
    ax.set_ylabel('N trades')
    ax.legend(title='Exit Reason')
    ax.tick_params(axis='x', rotation=15)

    plt.tight_layout()
    plt.show()
else:
    print('Sin trades — omitiendo visualizaciones.')

## 8. Análisis por Hora de Entrada

In [ ]:
if not all_trades.empty and not tradeable.empty:
    # Adjuntar hora de la barra de señal
    # Para cada trade, recuperar la hora de la barra de señal desde tradeable
    trade_hours = []
    for _, trade in all_trades.iterrows():
        day_rows = tradeable[
            (tradeable['ticker'] == trade['ticker']) &
            (tradeable['date'] == trade['date'])
        ].reset_index(drop=True)
        sig_bar = int(trade.get('signal_bar', 0))
        if sig_bar < len(day_rows):
            h = day_rows.iloc[sig_bar]['timestamp'].hour + day_rows.iloc[sig_bar]['timestamp'].minute / 60
        else:
            h = np.nan
        trade_hours.append(h)

    all_trades_h = all_trades.copy()
    all_trades_h['entry_hour'] = trade_hours
    all_trades_h = all_trades_h.dropna(subset=['entry_hour'])

    # Agrupar en bins de 30 min
    all_trades_h['hour_bin'] = pd.cut(
        all_trades_h['entry_hour'],
        bins=[9.5, 10.0, 10.5, 11.0, 11.5, 12.0, 13.0, 14.0, 15.0, 16.0],
        labels=['9:30', '10:00', '10:30', '11:00', '11:30', '12:00', '13:00', '14:00', '15:00'],
        right=False
    )

    hour_stats = all_trades_h.groupby(['strategy', 'hour_bin']).agg(
        n=('r_multiple', 'count'),
        avg_r=('r_multiple', 'mean'),
        win_rate=('r_multiple', lambda x: (x > 0).mean()),
    ).reset_index()

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    ax = axes[0]
    for strat in all_trades_h['strategy'].unique():
        sub = hour_stats[hour_stats['strategy'] == strat]
        ax.plot(sub['hour_bin'].astype(str), sub['avg_r'], marker='o', label=strat, linewidth=2)
    ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax.set_title('Avg R por Hora de Entrada')
    ax.set_xlabel('Hora ET')
    ax.set_ylabel('Avg R')
    ax.legend()
    ax.tick_params(axis='x', rotation=30)
    ax.grid(True, alpha=0.3)

    ax = axes[1]
    for strat in all_trades_h['strategy'].unique():
        sub = hour_stats[hour_stats['strategy'] == strat]
        ax.plot(sub['hour_bin'].astype(str), sub['win_rate'], marker='s', label=strat, linewidth=2)
    ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
    ax.set_title('Win Rate por Hora de Entrada')
    ax.set_xlabel('Hora ET')
    ax.set_ylabel('Win Rate')
    ax.set_ylim(0, 1)
    ax.legend()
    ax.tick_params(axis='x', rotation=30)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print('Análisis por hora omitido (sin datos).')

## 9. Top Trades y Worst Trades

In [ ]:
if not all_trades.empty:
    display_cols = ['ticker', 'date', 'strategy', 'dv_ratio_at_signal',
                    'ratio_bucket', 'entry_price', 'exit_price', 'exit_reason',
                    'r_multiple', 'net_pnl', 'hold_bars']
    available_cols = [c for c in display_cols if c in all_trades.columns]

    print('=== TOP 10 TRADES (mayor R) ===')
    display(all_trades[available_cols].sort_values('r_multiple', ascending=False).head(10).round(3))

    print('\n=== WORST 10 TRADES (menor R) ===')
    display(all_trades[available_cols].sort_values('r_multiple', ascending=True).head(10).round(3))

    print('\n=== RESUMEN GENERAL ===')
    total_pnl = all_trades['net_pnl'].sum()
    overall_wr = (all_trades['r_multiple'] > 0).mean()
    overall_r  = all_trades['r_multiple'].mean()
    print(f'Total trades: {len(all_trades)}')
    print(f'Win rate global: {overall_wr:.1%}')
    print(f'Avg R global: {overall_r:+.3f}')
    print(f'P&L total ({SHARES_PER_TRADE} shares/trade): ${total_pnl:+,.0f}')
else:
    print('Sin trades.')

## 10. Estado de los Datos — Diagnóstico

Si el backtest devuelve 0 trades, este bloque ayuda a entender por qué.

In [ ]:
print('=== DIAGNÓSTICO DE DATOS ===')
print()
print(f'DB path: {FINVIZ_DB}')
print(f'DB existe: {os.path.exists(FINVIZ_DB)}')

if os.path.exists(FINVIZ_DB):
    with sqlite3.connect(FINVIZ_DB) as conn:
        tables = [r[0] for r in conn.execute(
            "SELECT name FROM sqlite_master WHERE type='table'"
        ).fetchall()]
        print(f'Tablas en DB: {tables}')

        if 'snapshots' in tables:
            snap_count = conn.execute("SELECT COUNT(*) FROM snapshots WHERE category='Top Gainers'").fetchone()[0]
            snap_days  = conn.execute(
                "SELECT DISTINCT substr(timestamp,1,10) FROM snapshots WHERE category='Top Gainers' ORDER BY 1 DESC LIMIT 10"
            ).fetchall()
            print(f'Snapshots Top Gainers: {snap_count}')
            print(f'Últimos 10 días con snapshots: {[r[0] for r in snap_days]}')

        if 'market_bars' in tables:
            bar_count = conn.execute("SELECT COUNT(*) FROM market_bars").fetchone()[0]
            bar_tickers = conn.execute(
                "SELECT ticker, COUNT(*) as n, MIN(bar_time), MAX(bar_time) FROM market_bars GROUP BY ticker ORDER BY n DESC LIMIT 20"
            ).fetchall()
            print(f'\nBarras en market_bars: {bar_count}')
            print('Tickers con más barras:')
            for row in bar_tickers:
                print(f'  {row[0]:8s}  {row[1]:5d} barras  {row[2][:10]} → {row[3][:10]}')
        else:
            print()
            print('WARN: tabla market_bars NO EXISTE')
            print('  Para descargar barras: activar USE_TWS=True con TWS corriendo')
            print('  O ejecutar: python python/fetch_market_bars.py (si existe)')

print()
if not candidates.empty:
    print(f'Day T candidates: {len(candidates)}')
    cand_with_bars = set()
    if not bars_df.empty:
        cand_with_bars = set(bars_df['ticker'].unique())
    candidates_with_bars = candidates[candidates['ticker'].isin(cand_with_bars)]
    print(f'Candidates con barras disponibles: {len(candidates_with_bars)} / {len(candidates)}')
    missing_bars = candidates[~candidates['ticker'].isin(cand_with_bars)]
    if not missing_bars.empty:
        print(f'Tickers sin barras ({len(missing_bars["ticker"].unique())} únicos):')
        print(' ', sorted(missing_bars['ticker'].unique())[:30])

---
## Conclusiones

### Interpretación del DV Ratio

- **DV_ratio < 0.3 en T+1**: el día T+1 lleva solo el 30% del DV del Day T. Mercado aún frío → posible continuación si el precio está fuerte.
- **DV_ratio 0.6–1.0 en T+1**: actividad comparable al Day T → mercado activo, señal ambigua.
- **DV_ratio > 1.0 en T+1**: el T+1 ya supera al Day T en DV → posible agotamiento si precio empieza a debilitarse.

### Criterios para validar el edge

| Criterio | Mínimo para operar en live |
|---|---|
| N trades | >= 30 por estrategia |
| Win Rate | >= 55% (long) / >= 45% (short con RR>2) |
| Expectancy | >= +0.15R |
| Profit Factor | >= 1.3 |
| p-value | < 0.05 |

### Limitaciones
1. **Datos limitados**: el universo depende de los días con snapshots en finviz-dashboard.
2. **Slippage real**: en small caps el slippage real puede ser 3-5x mayor que el modelado.
3. **Short selling**: muchos small caps no tienen shares disponibles para short. Exhaustion Short puede ser solo teórico.
4. **Survivorship bias**: solo tenemos tickers que aparecieron en Top Gainers (ya son winners del Day T).

### Próximos pasos
1. Acumular más datos (market_bars) para los tickers del universo.
2. Segmentar por régimen de mercado (SPY > / < SMA20).
3. Optimizar filtros de señal: testar variantes de DV_ratio threshold.
4. Validar con paper trading en live antes de implementar en V5.